# CareMind: MedGemma chest X-ray experiment
Run MedGemma 1.5 4B on a Colab GPU. This notebook is an interactive experiment, not a hosted CareMind endpoint or a validated diagnostic tool. It handles a single PNG/JPEG chest X-ray; it does not process DICOM, MRI/CT volumes, or PDFs.

1. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Sign into [Hugging Face](https://huggingface.co/google/medgemma-1.5-4b-it) and accept/request the model's access terms.
3. Create a read token with access to that gated model. Upload a dedicated `.env` containing `HF_TOKEN=your_token` through Colab's Files sidebar into `/content/.env`. Alternatively, use Colab Secrets or the hidden prompt. Do not upload your project's full `.env` with other credentials.
4. Run cells in order. The first run downloads several GB. Free GPU availability and session duration vary.
5. Start with the supplied public image. Use only public or appropriately de-identified examples for your own experiments.

Based on [Google's model documentation](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card).
Quantization reduces memory use but can change output quality. Outputs need human review.


In [ ]:
%pip -q install "transformers>=4.50.3,<5" "accelerate>=1.5,<2" "bitsandbytes>=0.45.3,<1" "huggingface_hub>=0.30,<1" "Pillow>=10,<12" "python-dotenv>=1,<2" requests


In [ ]:
import gc
import io
import json
import os
import time
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
import torch
try:
    from google.colab import userdata, files
except ImportError:
    userdata = files = None
from huggingface_hub import login
from dotenv import dotenv_values
from PIL import Image
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. Select Runtime > Change runtime type > GPU and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
# Search the kernel's current directory, then its parents.
# A remote kernel cannot read files on your laptop automatically.
ENV_PATH = next((folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
                 if (folder / ".env").is_file()), Path("/content/.env"))
hf_token = None
if ENV_PATH.is_file():
    hf_token = dotenv_values(ENV_PATH, interpolate=False).get("HF_TOKEN")
hf_token = (hf_token or os.environ.get("HF_TOKEN") or "").strip()
if not hf_token and userdata is not None:
    try:
        hf_token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        hf_token = None
if not hf_token:
    hf_token = getpass("Hugging Face read token (hidden): ")
try:
    login(token=hf_token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = hf_token
finally:
    del hf_token


## Load the model
4-bit NF4 loading is enabled to reduce GPU memory use. A T4 uses float16 computation; newer GPUs use bfloat16 when supported. This cell may take several minutes. If you see a 401/403 error, check model approval and your token's gated-repository permissions. After a dependency/import error, restart the session and rerun the cells.


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
# Release an earlier model if this cell is rerun.
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    torch_dtype=compute_dtype,
    device_map={"": 0},
)
model.eval()
print("Model ready:", MODEL_ID)


## Choose an image
Leave `USE_PUBLIC_SAMPLE = True` for the first run. The sample is **Stillwaterising, CC0, via Wikimedia Commons**, also used in Google's tutorial. Set it to `False` to select one PNG/JPEG from your computer. Changing the image clears the previous result.


In [ ]:
import requests

USE_PUBLIC_SAMPLE = True
result = None
if USE_PUBLIC_SAMPLE:
    source = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
    response = requests.get(source, headers={"User-Agent": "CareMind-Colab-Experiment/1.0"}, timeout=60)
    response.raise_for_status()
    image_bytes = response.content
else:
    if files is not None:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one PNG or JPEG chest X-ray.")
        filename, image_bytes = next(iter(uploaded.items()))
    else:
        image_bytes = Path(input("Image path on the kernel machine: " )).expanduser().read_bytes()
    source = "user_upload"  # Avoid including a potentially identifying filename in exports.

with Image.open(io.BytesIO(image_bytes)) as opened:
    if opened.format not in {"PNG", "JPEG"}:
        raise ValueError("Only PNG/JPEG are supported here; DICOM and volumes need a separate pipeline.")
    xray = opened.convert("RGB")
print("Image size:", xray.size)
preview = xray.copy()
preview.thumbnail((768, 768))
display(preview)


## Analyze the image
Edit the question if desired. This generates unverified observations, not ground-truth findings or calibrated confidence scores. A single successful run does not establish accuracy.


In [ ]:
QUESTION = (
    "Describe the visible findings in this chest X-ray for a research demonstration. "
    "Organize your response into image quality, observations, and limitations. "
    "State uncertainty and do not invent patient history or comparison studies. "
    "Do not provide treatment recommendations."
)
result = None
messages = [{"role": "user", "content": [
    {"type": "image", "image": xray},
    {"type": "text", "text": QUESTION},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device, dtype=compute_dtype)
input_length = inputs["input_ids"].shape[-1]
started = time.perf_counter()
try:
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    answer = processor.decode(generated[0][input_length:], skip_special_tokens=True)
    result = {
        "schema_version": 1,
        "experiment": "caremind_colab_chest_xray",
        "model": MODEL_ID,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "quantization": "bitsandbytes_nf4_4bit",
        "gpu": torch.cuda.get_device_name(0),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source": source,
        "question": QUESTION,
        "answer": answer,
        "elapsed_seconds": round(time.perf_counter() - started, 2),
        "clinically_validated": False,
    }
    print("Experimental model output — requires review\n")
    print(answer)
except torch.cuda.OutOfMemoryError:
    print("GPU memory exhausted. Restart the session, rerun, and try fewer output tokens.")
    raise
finally:
    del inputs
    if "generated" in globals():
        del generated
    gc.collect()
    torch.cuda.empty_cache()


## Download the result
This exports model output and run metadata as JSON. It does not automatically upload anything to CareMind or turn generated claims into trusted evidence. Review output before sharing. Download it before the Colab runtime expires.


In [ ]:
if result is None:
    raise RuntimeError("Run the analysis cell successfully before exporting.")
output_path = (Path("/content") if files is not None else Path.cwd()) / "caremind_medgemma_result.json"
output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
if files is not None:
    files.download(str(output_path))
else:
    print("Result saved to:", output_path)


## Next steps and limits
- Compare generated observations against expert-reviewed reference reports on multiple public examples.
- The local CareMind ImagingAgent still uses paired reports/captions; this notebook does not enable live pixel analysis in the app.
- MRI/CT require study-specific preprocessing and evaluation, beyond this single-image experiment.
- For a live app, use a suitable hosted GPU endpoint after evaluation. Free Colab is intended for interactive notebooks, with limits on serving external interfaces: [Colab FAQ](https://research.google.com/colaboratory/faq.html).
- Stop the GPU session using **Runtime → Disconnect and delete runtime** when finished.
